### Block 0 – Install dependencies

In [1]:
!pip install -q datasets huggingface_hub scikit-learn tqdm pandas requests


### Block 1 – Load FiQA-SA (TheFinAI/flare-fiqasa, test split 235)

In [2]:
import os
import getpass
from datasets import load_dataset
from huggingface_hub import HfApi, HfFolder

DATASET_ID = "TheFinAI/flare-fiqasa"

# Get / reuse HF token
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf_...): ")

api = HfApi()
_ = api.dataset_info(DATASET_ID, token=hf_token)  # verify access
HfFolder.save_token(hf_token)
print(f"HF access to {DATASET_ID} OK ✅")

ds_all = load_dataset(DATASET_ID, token=hf_token)

print("Available splits and sizes:")
for split_name in ds_all.keys():
    print(f"  {split_name}: {len(ds_all[split_name])}")

dataset = ds_all["test"]
print("\nUsing split: test")
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Hugging Face token (hf_...): ··········
HF access to TheFinAI/flare-fiqasa OK ✅


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Available splits and sizes:
  train: 750
  test: 235
  valid: 188

Using split: test
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 2 – Shared helpers（label space, prompt builder, label extractor）

In [3]:
import re
from typing import List, Optional

from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

# Fixed label space for FiQA-SA
SENTIMENT_CHOICES = ["negative", "neutral", "positive"]


def build_prompt(text: str, choices: List[str] = SENTIMENT_CHOICES) -> str:
    """
    Build a multiple-choice style sentiment prompt for Grok-4.
    """
    return (
        "Analyze the sentiment of this financial text from an investor's perspective.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(choices)}\n"
        "Label:"
    )


def extract_label(
    model_text: Optional[str],
    choices: List[str] = SENTIMENT_CHOICES
) -> Optional[str]:
    """
    Extract one of the choices from the model output.
    First try exact match, then whole-word match.
    Returns the matched label as a string, or None if nothing is found.
    """
    if not model_text:
        return None

    s = model_text.strip().lower()

    # 1) exact match
    for c in choices:
        c_norm = c.strip().lower()
        if s == c_norm:
            return c

    # 2) whole-word match anywhere in the output
    for c in choices:
        c_norm = c.strip().lower()
        pat = r"\b" + re.escape(c_norm) + r"\b"
        if re.search(pat, s):
            return c

    # no valid label found
    return None


### Block 3 – Grok-4 client + call with retries

In [4]:
import os
import time
import json
import requests
from typing import Tuple, Dict, Any

GROK_MODEL_ID = "grok-4"  # adjust if your actual model id is different
BASE_URL = "https://api.x.ai/v1/chat/completions"  # check against Grok docs


# Get Grok API key
grok_api_key = os.getenv("GROK_API_KEY")
if not grok_api_key:
    import getpass
    grok_api_key = getpass.getpass("Grok API key (sk-...): ")

headers = {
    "Authorization": f"Bearer {grok_api_key}",
    "Content-Type": "application/json",
}


def call_grok_with_retries(
    prompt: str,
    model_id: str = GROK_MODEL_ID,
    max_retries: int = 6,
    sleep_base: float = 1.0,
) -> Tuple[Optional[str], Dict[str, Any]]:
    """
    Call Grok-4 via HTTP with simple exponential backoff.
    Returns (output_text, meta_dict).
    NOTE: Adjust BASE_URL and JSON parsing according to Grok's official docs.
    """
    last_err = None

    for attempt in range(max_retries):
        try:
            payload = {
                "model": model_id,
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                # you can add more parameters if Grok supports them, e.g. temperature
                "temperature": 0.0,
            }

            resp = requests.post(
                BASE_URL,
                headers=headers,
                data=json.dumps(payload),
                timeout=60,
            )
            if resp.status_code != 200:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")

            data = resp.json()

            # NOTE: Adjust the path if Grok uses a different schema.
            # This matches many OpenAI-style chat APIs:
            text_out = (
                data.get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", None)
            )

            usage = data.get("usage", None)
            meta = {
                "raw_status": resp.status_code,
                "id": data.get("id", None),
                "usage": usage,
            }
            return text_out, meta

        except Exception as e:
            last_err = str(e)
            wait = sleep_base * (2 ** attempt)
            print(f"[Grok-4] Error on attempt {attempt+1}/{max_retries}: {last_err}")
            time.sleep(wait)

    return None, {"error": last_err}


Grok API key (sk-...): ··········


### Block 4 – Evaluation loop on FiQA-SA (Grok-4 only)

In [5]:
def evaluate_grok4_on_fiqasa(dataset) -> pd.DataFrame:
    """
    Evaluate Grok-4 on a FiQA-SA style dataset with fields:
      - text   : the input sentence
      - answer : gold label in {'negative','neutral','positive'}

    Returns a pandas DataFrame with per-example results, and prints metrics.
    """
    tag = "grok-4"
    records = []

    print(f"\n=== Evaluating model: {tag} on {len(dataset)} FiQA-SA examples ===\n")

    for i, ex in enumerate(tqdm(dataset, total=len(dataset), desc=tag)):
        text = ex["text"]
        gold_label_str = ex["answer"].strip().lower()

        if gold_label_str not in SENTIMENT_CHOICES:
            continue

        gold_idx = SENTIMENT_CHOICES.index(gold_label_str)

        prompt = build_prompt(text, SENTIMENT_CHOICES)
        model_out, meta = call_grok_with_retries(prompt)

        pred_label = extract_label(model_out, SENTIMENT_CHOICES)
        if pred_label is None:
            pred_idx = -1
        else:
            pred_idx = SENTIMENT_CHOICES.index(pred_label)

        rec = {
            "row_index": i,
            "text": text,
            "gold_label": gold_label_str,
            "gold_index": gold_idx,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold_idx),
            "meta": meta,
        }
        records.append(rec)

    df = pd.DataFrame(records)

    if len(df) == 0:
        print("No examples were evaluated.")
        return df

    valid_mask = df["predicted_index"] >= 0

    acc_all = accuracy_score(df["gold_index"], df["predicted_index"])
    acc_valid = (
        accuracy_score(df.loc[valid_mask, "gold_index"], df.loc[valid_mask, "predicted_index"])
        if valid_mask.any()
        else None
    )

    print(f"\n[{tag}] Accuracy (all examples): {acc_all:.4f}")
    if acc_valid is not None:
        print(f"[{tag}] Accuracy (valid predictions only): {acc_valid:.4f}")
    else:
        print(f"[{tag}] No valid predictions (predicted_index >= 0).")

    if valid_mask.any():
        print(f"\n[{tag}] Classification report (valid predictions only):")
        print(
            classification_report(
                df.loc[valid_mask, "gold_index"],
                df.loc[valid_mask, "predicted_index"],
                labels=list(range(len(SENTIMENT_CHOICES))),
                target_names=SENTIMENT_CHOICES,
            )
        )

    print(f"\n[{tag}] Prediction distribution (including invalid=-1):")
    print(df["predicted_index"].value_counts().sort_index())

    return df


# Run evaluation
df_grok4 = evaluate_grok4_on_fiqasa(dataset)

# Optionally save predictions
df_grok4.to_csv("fiqasa_grok4_predictions.csv", index=False)
print("Saved predictions to: fiqasa_grok4_predictions.csv")



=== Evaluating model: grok-4 on 235 FiQA-SA examples ===



grok-4: 100%|██████████| 235/235 [44:55<00:00, 11.47s/it]


[grok-4] Accuracy (all examples): 0.8511
[grok-4] Accuracy (valid predictions only): 0.8511

[grok-4] Classification report (valid predictions only):
              precision    recall  f1-score   support

    negative       0.88      0.86      0.87        76
     neutral       0.33      0.33      0.33        18
    positive       0.90      0.91      0.91       141

    accuracy                           0.85       235
   macro avg       0.70      0.70      0.70       235
weighted avg       0.85      0.85      0.85       235


[grok-4] Prediction distribution (including invalid=-1):
predicted_index
0     74
1     18
2    143
Name: count, dtype: int64
Saved predictions to: fiqasa_grok4_predictions.csv
